# RoSE + query decomposition, Algorithm 1 — ChartQAPro factoid

Evaluates **Algorithm 1 only** on the first `N_QUESTIONS` factoid
questions in dataset order (`factoid-00000` …) — the same window as the
routed run, so `results/compare_arms.py` can line the arms up by id.
Algorithm 2 has its own notebook (`decomp_alg2_chartqapro.ipynb`);
run them in separate sessions.

**Algorithm 1 — decompose → parallel answer → evidence → RoSE.** The 7B
splits the question into two sub-queries (JSON; `[question, question]`
fallback on parse failure). Gemma 3 4B answers sub-query 1 and Qwen3.5-4B
answers sub-query 2, each with 3 sampled paths and the agreement ratio as
confidence. The sub-answers are prepended to the RoSE reasoning prompt as
an evidence block; the final answer comes from RoSE's own vote on
**Qwen2.5-VL-7B**. No routing anywhere.

**FAST_MODE (default on).** Four notebook-level overrides cut the
per-question cost roughly in half without touching the repo's code:
RoSE samples **3** reasoning paths instead of 5, the two-stage
chart-description pass and CLIP hybrid retrieval are off, images are
capped at 1024·28·28 pixels, and the decomposition call gets 128 tokens.
The run's `*_meta.json` records the exact configuration either way, so a
fast run can never be mistaken for a spec-exact one. Set
`FAST_MODE = False` to reproduce the full configuration used by the
routed run (m=5, two-stage on, CLIP on).

**Runtime (T4 x2).** With the default `FAST_MODE = True` ≈ **4–6 min per
question** → ~19–25 h at N=250, i.e. 2–3 budgeted sessions chained via
`RESUME_FROM`. Spec-exact mode (`FAST_MODE = False`) ≈ 7–8 min/q →
~29–33 h. For a single-session result, set `N_QUESTIONS = 100` (~8–10 h
fast): the run is a dataset-order prefix, so it stays id-aligned with
the routed run and `results/compare_arms.py` still works.

Every question is checkpointed, a stopped run publishes all outputs, and
errored questions are retried on resume — a **Save & Run All** version
with `SESSION_BUDGET_H = 10.0` always finishes inside the 12 h cap.


In [ ]:
# ── config ───────────────────────────────────────────────────────
ALGORITHM   = 1             # this notebook runs Algorithm 1 only
DATASET_KEY = "chartqapro"
N_QUESTIONS = 250         # first N factoid questions in dataset order;
                          # lower it for a single-session run (any prefix
                          # stays comparable across arms)
FAST_MODE   = True        # notebook-level speed preset (see the cell
                          # below for exactly what it changes); False =
                          # the spec-exact config of the routed run
DRY_RUN     = False       # True = answer only the first 2 questions
QUANT       = "nf4"       # same quantization as the pilot + routed runs
SESSION_BUDGET_H = 10.0   # stop cleanly after this many hours so a
                          # "Save & Run All" version FINISHES inside the
                          # T4 x2's 12 h session cap and publishes the
                          # checkpoint; None = no budget

REPO_URL  = "https://github.com/ktahsinr/RoSE.git"
REPO_REF  = "feature/decompose-vote"

# Chaining sessions: download decomp_alg1_checkpoint.json from a
# finished version's Output pane, attach it (Add Input → Your Work →
# this notebook, or upload as a dataset) and point at it here — answered
# questions are skipped, errored ones retried.
RESUME_FROM = None

import os, time
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # Kaggle can't render the widgets
T_START = globals().get("T_START", time.time())    # budget clock survives cell re-runs
from pathlib import Path
import sys
try:
    sys.stdout.reconfigure(line_buffering=True)    # live logs in Kaggle's pane
except (AttributeError, ValueError):
    pass

ON_KAGGLE   = Path("/kaggle").exists()
WORK        = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
SCRATCH     = Path("/kaggle/temp")    if ON_KAGGLE else WORK / "scratch"
CODE_DIR    = SCRATCH / "RoSE"
IMAGES_DIR  = SCRATCH / "images"
RESULTS_DIR = WORK / "results" / "decompose"       # published with the notebook
DATASET_JSON = SCRATCH / "chartqapro_factoid_decomp.json"
OUT_FILE     = RESULTS_DIR / f"decomp_alg{ALGORITHM}.json"

for d in (SCRATCH, IMAGES_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"algorithm={ALGORITHM}  n={N_QUESTIONS}  fast={FAST_MODE}  "
      f"dry_run={DRY_RUN}  quant={QUANT}")


In [ ]:
# ── code + pinned libraries ──────────────────────────────────────
import subprocess
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                    REPO_URL, str(CODE_DIR)], check=True)
else:
    # a kernel restart keeps /kaggle/temp — refresh so fixes pushed to the
    # branch mid-session are picked up instead of running stale code
    subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1",
                    "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "reset", "--hard",
                    "FETCH_HEAD"], check=True)
print("repo @", subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(CODE_DIR / "requirements-pinned.txt")], check=True)

# The sweep pin (transformers==4.51.3) supports Gemma 3 and Qwen2.5-VL
# but NOT Qwen3.5 (model_type "qwen3_5" → ValueError at load time). Probe
# for it in a FRESH interpreter — before transformers is imported in this
# kernel, so an upgrade can never leave a stale module behind — and
# upgrade only if needed. The run's *_meta.json records the exact version
# used either way, so the report stays honest.
def _knows_qwen35():
    return subprocess.run(
        [sys.executable, "-c",
         "from transformers.models.auto.configuration_auto import "
         "CONFIG_MAPPING; import sys; "
         "sys.exit(0 if 'qwen3_5' in CONFIG_MAPPING else 1)"],
    ).returncode == 0

if not _knows_qwen35():
    if "transformers" in sys.modules:
        raise SystemExit(
            "transformers needs an upgrade for Qwen3.5 but is already "
            "imported in this kernel — restart the kernel and Run All.")
    print("pinned transformers lacks the qwen3_5 architecture — "
          "upgrading to the latest release…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers"], check=True)
    if not _knows_qwen35():
        print("latest release still lacks qwen3_5 — installing from source…")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "git+https://github.com/huggingface/transformers.git"],
            check=True)
        if not _knows_qwen35():
            raise SystemExit(
                "no available transformers build recognizes qwen3_5 — "
                "check https://huggingface.co/Qwen/Qwen3.5-4B for the "
                "version it requires")

if str(CODE_DIR / "chartqapro") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "chartqapro"))
import transformers
print("transformers:", transformers.__version__)


In [ ]:
# ── HuggingFace auth + gated-access check (all three models) ─────
# Every question touches all three models, so access to every one is
# verified up front — better a failure here than hours in.
# check_gated_access DOWNLOADS each config.json, the exact request the
# model load makes (a gated repo's metadata is public, so model_info()
# "succeeds" with no token and the run still 401s at load time).
NO_TOKEN_HELP = (
    "→ No HF token reached this session:\n"
    "  1. hf.co → Settings → Access Tokens → create a READ token\n"
    "     (use the SAME account that accepted the Gemma 3 license)\n"
    "  2. Kaggle → Add-ons → Secrets → add it with label HF_TOKEN\n"
    "  3. TICK THE CHECKBOX attaching the secret to THIS notebook —\n"
    "     the attachment is per-notebook and defaults to off\n"
    "  then re-run this cell.\n")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"could not read Kaggle secret HF_TOKEN: {e}")

if token:
    from huggingface_hub import login
    login(token=token)
    print("HF login ok")
elif ON_KAGGLE:
    raise SystemExit("\nNO HF TOKEN.\n" + NO_TOKEN_HELP)

from decompose.models import check_gated_access
check_gated_access()


In [ ]:
# ── build the first-N ChartQAPro factoid set ─────────────────────
# Ids are factoid ordinals in split order (factoid-00000 …), the repo's
# convention — the SAME window the routed run used, so compare_arms.py
# lines the arms up by id afterwards. No sampling; only these questions
# get their images decoded. Skipped instantly if already built this
# session (a kernel restart keeps /kaggle/temp).
import io, json, ast

if DATASET_JSON.exists() and len(json.loads(DATASET_JSON.read_text())) >= N_QUESTIONS:
    print(f"dataset already built → {DATASET_JSON}")
else:
    from PIL import Image
    from datasets import load_dataset

    def _gold(v):
        if isinstance(v, (list, tuple)):
            return [str(x) for x in v]
        s = str(v).strip()
        if s.startswith("["):
            try:
                p = ast.literal_eval(s)
                if isinstance(p, (list, tuple)):
                    return [str(x) for x in p]
            except Exception:
                pass
        return s

    def _save_image(field, path):
        if isinstance(field, dict) and field.get("bytes"):
            img = Image.open(io.BytesIO(field["bytes"]))
        elif isinstance(field, (bytes, bytearray)):
            img = Image.open(io.BytesIO(field))
        elif isinstance(field, Image.Image):
            img = field
        else:
            return str(field)
        img.convert("RGB").save(path, format="PNG")
        return str(path)

    ds = load_dataset("ahmed-masry/ChartQAPro", split="test")
    qtypes = [str(t).strip().lower() for t in ds["Question Type"]]
    fact_idx = [i for i, t in enumerate(qtypes) if t == "factoid"]
    print(f"factoid questions in split: {len(fact_idx)}")

    take = min(N_QUESTIONS, len(fact_idx))
    records = []
    for idx in range(take):                  # idx = factoid ordinal = id
        row = ds[fact_idx[idx]]
        records.append({
            "id":            f"factoid-{idx:05d}",
            "question":      str(row.get("Question", "")).strip(),
            "choices":       None,
            "answer":        _gold(row.get("Answer")),
            "question_type": "factoid",
            "image":         _save_image(row.get("image"),
                                         IMAGES_DIR / f"{idx:05d}.png"),
        })

    DATASET_JSON.write_text(json.dumps(records, ensure_ascii=False, indent=2))
    print(f"first {len(records)} factoid questions + images → {DATASET_JSON}")


In [ ]:
# ── RoSE config + the FAST_MODE preset ───────────────────────────
# Every value is SET explicitly (not asserted), so re-running this cell
# is always safe. FAST_MODE trims the per-question cost without touching
# the repo's code; the run's meta JSON records whatever was used.
import rose_chartqapro as rose
import decompose.decomp_runner as decomp_runner

assert (rose.K_DEMONSTRATIONS, rose.LAMBDA, rose.TEMPERATURE) == (3, 1.2, 0.8), \
    "rose_chartqapro defaults drifted from the sweep config"
assert not rose.PAPER_FAITHFUL
rose.IMAGES_DIR = str(IMAGES_DIR)

# m sampled reasoning paths: 3 is the paper-adjacent floor (greedy path
# + 2 samples) and saves ~2 full 7B generations per question.
rose.M_PATHS = 3 if FAST_MODE else 5
# The T=0 chart-description pass costs ~25-30 s/question on its own.
rose.EXTENSIONS["two_stage_reasoning"] = not FAST_MODE
# CLIP hybrid retrieval: skipped in fast mode (CPU, but adds per-question
# work and a model download).
rose.EXTENSIONS["visual_hybrid_retrieval"] = not FAST_MODE
# Smaller vision grids = faster prefill on every generation.
rose.MAX_PIXELS = (1024 if FAST_MODE else 1280) * 28 * 28
# A two-string JSON list never needs more than ~128 tokens.
decomp_runner.DECOMP_MAX_TOKENS = 128 if FAST_MODE else 256

print(f"{'FAST' if FAST_MODE else 'SPEC-EXACT'} config:  "
      f"m={rose.M_PATHS}  k={rose.K_DEMONSTRATIONS}  λ={rose.LAMBDA}  "
      f"T={rose.TEMPERATURE}  max_pixels={rose.MAX_PIXELS}  "
      f"decomp_tokens={decomp_runner.DECOMP_MAX_TOKENS}")
for name, on in rose.active_extensions().items():
    print(f"   {'✓' if on else '·'} {name}")

# The one embedder for everything: RoSE demo retrieval, the 7B pool, and
# (Algorithm 2) the decomposition vote. The HF Hub sometimes throws
# transient 500s: use the local cache when it is complete, otherwise
# retry the download with long backoff instead of dying.
from sentence_transformers import SentenceTransformer

def _load_embedder(name, attempts=6):
    try:                       # a complete local cache needs no Hub at all
        return SentenceTransformer(name, local_files_only=True)
    except TypeError:          # older sentence-transformers: no such kwarg
        pass
    except Exception:
        pass                   # not cached yet — download below
    last = None
    for i in range(1, attempts + 1):
        try:
            return SentenceTransformer(name)
        except Exception as e:
            last = e
            wait = min(45 * i, 240)
            print(f"embedder download failed ({type(e).__name__}) — "
                  f"retry {i}/{attempts} in {wait}s")
            time.sleep(wait)
    raise SystemExit(
        f"could not load {name} after {attempts} attempts — the HF Hub is "
        f"having problems (last error: {last}). Re-run this cell in a few "
        "minutes; everything already downloaded is cached.")

if "embedder" not in globals():
    embedder = _load_embedder(rose.EMBED_MODEL)
print(f"embedder: {rose.EMBED_MODEL}")

import shutil
if RESUME_FROM:
    ckpt = OUT_FILE.with_name(OUT_FILE.stem + "_checkpoint.json")
    shutil.copy(RESUME_FROM, ckpt)
    print(f"resume checkpoint staged → {ckpt}")


In [ ]:
# ── run Algorithm 1 (GPU) ────────────────────────────────────────
# All three 4-bit VLMs are loaded ONCE and stay resident (≈13 GB — needs
# the T4 x2 accelerator); a kernel re-run reuses them instead of
# reloading. Questions run in dataset order with a checkpoint after
# EVERY question, so a partial run is an unbiased prefix and nothing is
# lost if the session dies. The session budget stops the run cleanly
# BETWEEN questions, so a headless "Save & Run All" version always
# publishes its outputs.
from decompose.decomp_runner import load_all_models, run_decomp

if "loaded" not in globals():
    loaded, revisions = load_all_models(QUANT)

remaining = None
if SESSION_BUDGET_H is not None:
    remaining = max(SESSION_BUDGET_H * 3600 - (time.time() - T_START), 60)

results = run_decomp(
    ALGORITHM, DATASET_JSON, OUT_FILE,
    n=N_QUESTIONS, quant=QUANT,
    embedder=embedder, loaded=loaded, revisions=revisions,
    limit=2 if DRY_RUN else None,
    stop_after_s=remaining, check_access=False)   # access verified above


In [ ]:
# ── report ───────────────────────────────────────────────────────
# The runner already wrote decomp_alg1.json (+ _checkpoint),
# decomp_alg1.csv (per question, rescored with scoring.py),
# decomp_alg1_stages.csv (per-stage latency/accuracy) and
# decomp_alg1_meta.json (revisions, versions, algorithm config).
import csv, json

meta = json.loads(OUT_FILE.with_name(OUT_FILE.stem + "_meta.json").read_text())
print(f"algorithm {meta['algorithm']}   answered {meta['n']}/{meta['n_total']}   "
      f"complete: {meta['complete']}")
print(f"rescored accuracy : {meta['accuracy_rescored']:5.1f}%")
if meta.get("mean_latency_s"):
    print(f"mean latency      : {meta['mean_latency_s']:6.1f} s/question")
print(f"params engaged    : {meta['params_b_used']:.1f} B/question")
if meta.get("vote_rules"):
    rules = "  ".join(f"{k}: {v}" for k, v in sorted(meta["vote_rules"].items()))
    print(f"vote rules        : {rules}")

print("\nper-stage summary:")
with (OUT_FILE.with_name(OUT_FILE.stem + "_stages.csv")).open() as f:
    for row in csv.DictReader(f):
        extra = "".join(
            f"  {k}={row[k]}" for k in ("mean_confidence", "accuracy_pct",
                                        "fallback_rate") if row.get(k))
        print(f"  {row['stage']:<22} n={row['n']:>4}  "
              f"{row['mean_latency_s']:>7}s{extra}")

print(f"\ndownload from the Output tab: {OUT_FILE.name}, "
      f"{OUT_FILE.stem}.csv, {OUT_FILE.stem}_stages.csv, "
      f"{OUT_FILE.stem}_meta.json, {OUT_FILE.stem}_checkpoint.json")
print("cross-arm table (offline):  cd chartqapro && python results/compare_arms.py")


## After the session

**Headless chaining (Save & Run All):** with `SESSION_BUDGET_H` set, each
committed version runs up to the budget, stops cleanly between questions,
and PUBLISHES its outputs. If `decomp_alg1_meta.json` says
`"complete": false`: Add Input → Your Work → this notebook (attaches the
latest version's output), set
`RESUME_FROM = "/kaggle/input/<slug>/results/decompose/decomp_alg1_checkpoint.json"`,
and Save & Run All again — answered questions are skipped, errored rows
retried. Re-pin the input to the newest version before each re-run.
**Keep `FAST_MODE` the same across chained sessions** so one run is one
configuration.

1. Download from the Output pane: `decomp_alg1.json`,
   `decomp_alg1.csv`, `decomp_alg1_stages.csv`,
   `decomp_alg1_meta.json` (and `_checkpoint.json` if unfinished).
2. Drop the finished files into `chartqapro/results/decompose/` in the
   repo. The meta file records the resolved model revisions, installed
   library versions, the FAST/spec configuration actually used, and the
   full algorithm config — a finished run is self-describing.
3. The cross-arm table (accuracy, mean params/q, mean latency/q, and the
   Alg 1 sub-answer-vs-final confusion counts) comes from
   `python results/compare_arms.py` inside `chartqapro/` — it rescores
   every arm with `scoring.py` on their common question window.
